# 04 - Model Validation (Synthetic Injection & Expert Review)

**Purpose:**
- Directly addresses **Reviewer C1** requesting stronger model evaluation.
- Unsupervised models lack ground truth, so we evaluate them in two ways:
  1. **Quantitative (Synthetic Outlier Injection):** Extract a known "clean" subset, inject severe synthetic outliers, and measure Precision, Recall, and F1 Score for IF and LOF.
  2. **Qualitative (Expert Face-Validity):** Isolate the highest-confidence anomalies (the IF $\cap$ LOF overlap set) and export the top 15 cases to a CSV template for manual clinical/expert annotation.

**Inputs:**
- `data/processed/features_raw.parquet`
- `data/processed/anomaly_labels.parquet`

**Outputs:**
- `outputs/tables/table_synthetic_validation.csv`
- `outputs/tables/table_expert_review_template.csv`\n

In [ ]:
# Cell 01: Mount Storage & Bootstrap Paths
import os
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Paper1_Revision')
else:
    BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

for folder in ['data/raw', 'data/interim', 'data/processed', 
               'outputs/figures', 'outputs/tables', 'outputs/models', 
               'outputs/notebook_exports']:
    (BASE_DIR / folder).mkdir(parents=True, exist_ok=True)
    
print(f"Base directory set to: {BASE_DIR}")\n

In [ ]:
# Cell 02: Imports, Global Seeds & Style
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import precision_score, recall_score, f1_score
import warnings

warnings.filterwarnings('ignore')

plt.style.use('default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'figure.dpi': 300,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.autolayout': True
})

# For reproducibility of synthetic generation
np.random.seed(42)\n

In [ ]:
# Cell 03: Load Features and Labels
features_path = BASE_DIR / 'data' / 'processed' / 'features_raw.parquet'
labels_path = BASE_DIR / 'data' / 'processed' / 'anomaly_labels.parquet'

df_features = pd.read_parquet(features_path)
df_labels = pd.read_parquet(labels_path)

print(f"Features shape: {df_features.shape}")
print(f"Labels shape: {df_labels.shape}")\n

## Part 1: Quantitative Validation via Synthetic Injection\n

In [ ]:
# Cell 04: Create a "Clean" Subset
# We assume points flagged as NORMAL by both IF and LOF are relatively clean.
clean_mask = (df_labels['IF_Label'] == 1) & (df_labels['LOF_05_Label'] == 1)
df_clean = df_features[clean_mask].copy()

print(f"Number of 'clean' records: {len(df_clean)}")

# Sample 10,000 clean records for our validation set
n_samples = 10000
df_val = df_clean.sample(n=n_samples, random_state=42).copy()

# True labels for validation set (all Normal = 1)
y_true = np.ones(n_samples)\n

In [ ]:
# Cell 05: Inject Synthetic Anomalies
# We will inject 500 anomalies (5% contamination)
n_anomalies = 500

# Randomly select 500 indices to become anomalies
anomaly_indices = np.random.choice(n_samples, size=n_anomalies, replace=False)

# Update ground truth: -1 for anomalies
y_true[anomaly_indices] = -1

# Perturb the features for the chosen anomalies
# We inject anomalies by shifting values by 3 to 5 standard deviations in random directions
std_devs = df_val.std()

for idx in anomaly_indices:
    for col in df_val.columns:
        # 50% chance to perturb a specific feature for an anomaly
        if np.random.rand() > 0.5:
            shift = np.random.uniform(3, 5) * std_devs[col]
            sign = np.random.choice([-1, 1])
            df_val.iloc[idx, df_val.columns.get_loc(col)] += (shift * sign)

# Ensure no negative values if the domain requires it (e.g. counts/amounts are usually >= 0)
df_val = df_val.clip(lower=0)

print(f"Validation set size: {len(df_val)}")
print(f"Injected anomalies: {n_anomalies}")\n

In [ ]:
# Cell 06: Evaluate Models on Validation Set
# Scale the validation set
scaler = StandardScaler()
X_val_scaled = scaler.fit_transform(df_val)

# IF
if_model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42, n_jobs=-1)
y_pred_if = if_model.fit_predict(X_val_scaled)

# LOF
lof_model = LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=False, n_jobs=-1)
y_pred_lof = lof_model.fit_predict(X_val_scaled)

# Since scikit-learn metrics expect 1 for positive class, we map Anomaly (-1) to 1, Normal (1) to 0
y_true_binary = (y_true == -1).astype(int)
y_pred_if_binary = (y_pred_if == -1).astype(int)
y_pred_lof_binary = (y_pred_lof == -1).astype(int)

res_if = {
    'Model': 'Isolation Forest',
    'Precision': precision_score(y_true_binary, y_pred_if_binary),
    'Recall': recall_score(y_true_binary, y_pred_if_binary),
    'F1-Score': f1_score(y_true_binary, y_pred_if_binary)
}

res_lof = {
    'Model': 'Local Outlier Factor',
    'Precision': precision_score(y_true_binary, y_pred_lof_binary),
    'Recall': recall_score(y_true_binary, y_pred_lof_binary),
    'F1-Score': f1_score(y_true_binary, y_pred_lof_binary)
}

df_synthetic_results = pd.DataFrame([res_if, res_lof])

print("\nSynthetic Validation Results:")
display(df_synthetic_results)

out_val = BASE_DIR / 'outputs' / 'tables' / 'table_synthetic_validation.csv'
df_synthetic_results.to_csv(out_val, index=False)
print(f"Exported to {out_val}")\n

## Part 2: Qualitative Validation via Expert Face-Validity\n

In [ ]:
# Cell 07: Extract Top Overlap Cases for Expert Review
# The manuscript states there are exactly 545 records flagged by BOTH models (IF and LOF-05)
overlap_mask = (df_labels['IF_Label'] == -1) & (df_labels['LOF_05_Label'] == -1)
df_overlap = df_features[overlap_mask].copy()

print(f"Overlap anomalies: {len(df_overlap)}")
assert len(df_overlap) == 545, f"Parity failure: Expected 545 overlap cases, got {len(df_overlap)}"

# To find the "most extreme" cases, we can sort by a proxy metric, such as 'Average Medicare Payment Amount'
# or simply take a random sample. We'll take the top 15 by Payment Amount.
df_top15 = df_overlap.sort_values(by='Average Medicare Payment Amount', ascending=False).head(15).copy()

# We add a blank column for the clinical expert to fill in their justification
df_top15['Expert_Clinical_Justification'] = ""

out_expert = BASE_DIR / 'outputs' / 'tables' / 'table_expert_review_template.csv'
df_top15.to_csv(out_expert, index=False)
print(f"Exported top 15 overlap template for manual review to: {out_expert}")
display(df_top15.head())\n